In [2]:
import pandas as pd
import re

In [3]:
df=pd.read_csv("flipkart_com-ecommerce_sample.csv")
#print(df.columns)
#rint(df.info())
#print(df.head(1))
df1=df["is_FK_Advantage_product"]
print(df1)

0        False
1        False
2        False
3        False
4        False
         ...  
19997    False
19998    False
19999    False
20000      NaN
20001      NaN
Name: is_FK_Advantage_product, Length: 20002, dtype: object


In [4]:
# noms_uniques = df['product_name'].unique()

# # On mesure la taille de la liste
# print(len(noms_uniques))
noms_uniques = df['product_name'].unique()
df_uniques = pd.DataFrame(noms_uniques, columns=['product_name_unique'])

# 3. Sauvegarder ce résultat dans un nouveau fichier CSV
# index=False évite d'ajouter une colonne de numéros inutile au début du fichier
#df_uniques.to_csv("produits_uniques.csv", index=False)

#print("Le fichier 'produits_uniques.csv' a été créé avec succès !")

In [6]:
df_important = df[["uniq_id", "product_name", "description", "product_category_tree", "brand"]]
print(df_important.isnull().sum())
df_important = df_important.fillna("")
print(df_important.isnull().sum())


uniq_id                     2
product_name                2
description                 4
product_category_tree       2
brand                    5866
dtype: int64
uniq_id                  0
product_name             0
description              0
product_category_tree    0
brand                    0
dtype: int64


In [7]:
df_important["document_text"] = (
    df_important["product_name"] + " " +
    df_important["description"] + " " +
    df_important["product_category_tree"] + " " +
    df_important["brand"]
)

In [8]:
print(df_important[["uniq_id", "product_name", "document_text"]].head())

                            uniq_id                           product_name  \
0  c2d766ca982eca8304150849735ffef9    Alisha Solid Women's Cycling Shorts   
1  7f7036a6d550aaa89d34c77bd39a5e48    FabHomeDecor Fabric Double Sofa Bed   
2  f449ec65dcbc041b6ae5e6a32717d01b                             AW Bellies   
3  0973b37acd0c664e3de26e97e5571454    Alisha Solid Women's Cycling Shorts   
4  bc940ea42ee6bef5ac7cea3fb5cfbee7  Sicons All Purpose Arnica Dog Shampoo   

                                       document_text  
0  Alisha Solid Women's Cycling Shorts Key Featur...  
1  FabHomeDecor Fabric Double Sofa Bed FabHomeDec...  
2  AW Bellies Key Features of AW Bellies Sandals ...  
3  Alisha Solid Women's Cycling Shorts Key Featur...  
4  Sicons All Purpose Arnica Dog Shampoo Specific...  


In [9]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    return text

In [10]:
df_important["clean_text"] = df_important["document_text"].apply(clean_text)

In [11]:
print(df_important[["document_text", "clean_text"]].head())

                                       document_text  \
0  Alisha Solid Women's Cycling Shorts Key Featur...   
1  FabHomeDecor Fabric Double Sofa Bed FabHomeDec...   
2  AW Bellies Key Features of AW Bellies Sandals ...   
3  Alisha Solid Women's Cycling Shorts Key Featur...   
4  Sicons All Purpose Arnica Dog Shampoo Specific...   

                                          clean_text  
0  alisha solid women s cycling shorts key featur...  
1  fabhomedecor fabric double sofa bed fabhomedec...  
2  aw bellies key features of aw bellies sandals ...  
3  alisha solid women s cycling shorts key featur...  
4  sicons all purpose arnica dog shampoo specific...  


In [12]:
def tokenize(text):
    return text.split()

In [14]:
df_important["tokens"] = df_important["clean_text"].apply(tokenize)
print(df_important[["clean_text", "tokens"]].head())

                                          clean_text  \
0  alisha solid women s cycling shorts key featur...   
1  fabhomedecor fabric double sofa bed fabhomedec...   
2  aw bellies key features of aw bellies sandals ...   
3  alisha solid women s cycling shorts key featur...   
4  sicons all purpose arnica dog shampoo specific...   

                                              tokens  
0  [alisha, solid, women, s, cycling, shorts, key...  
1  [fabhomedecor, fabric, double, sofa, bed, fabh...  
2  [aw, bellies, key, features, of, aw, bellies, ...  
3  [alisha, solid, women, s, cycling, shorts, key...  
4  [sicons, all, purpose, arnica, dog, shampoo, s...  


In [15]:
stop_words = {
    "the", "and", "for", "with", "of", "in", "on", "a", "an",
    "to", "is", "are", "this", "that", "by", "from", "as", "at",
    "it", "be", "or", "your", "you"
}

In [16]:
def remove_stop_words(tokens):
    return [word for word in tokens if word not in stop_words]

In [17]:
df_important["tokens"] = df_important["tokens"].apply(remove_stop_words)

In [18]:
def remove_short_words(tokens):
    return [word for word in tokens if len(word) > 1]

In [19]:
df_important["tokens"] = df_important["tokens"].apply(remove_short_words)

In [20]:
df_important["final_text"] = df_important["tokens"].apply(lambda tokens: " ".join(tokens))

In [21]:
print(df_important[["uniq_id", "product_name", "final_text"]].head())

                            uniq_id                           product_name  \
0  c2d766ca982eca8304150849735ffef9    Alisha Solid Women's Cycling Shorts   
1  7f7036a6d550aaa89d34c77bd39a5e48    FabHomeDecor Fabric Double Sofa Bed   
2  f449ec65dcbc041b6ae5e6a32717d01b                             AW Bellies   
3  0973b37acd0c664e3de26e97e5571454    Alisha Solid Women's Cycling Shorts   
4  bc940ea42ee6bef5ac7cea3fb5cfbee7  Sicons All Purpose Arnica Dog Shampoo   

                                          final_text  
0  alisha solid women cycling shorts key features...  
1  fabhomedecor fabric double sofa bed fabhomedec...  
2  aw bellies key features aw bellies sandals wed...  
3  alisha solid women cycling shorts key features...  
4  sicons all purpose arnica dog shampoo specific...  


In [23]:
print("Nom du produit :")
print(df_important.loc[0, "product_name"])

print("\nTexte original :")
print(df_important.loc[0, "document_text"])

print("\nTexte nettoyé :")
print(df_important.loc[0, "clean_text"])

print("\nTokens :")
print(df_important.loc[0, "tokens"])

print("\nTexte final :")
print(df_important.loc[0, "final_text"])

Nom du produit :
Alisha Solid Women's Cycling Shorts

Texte original :
Alisha Solid Women's Cycling Shorts Key Features of Alisha Solid Women's Cycling Shorts Cotton Lycra Navy, Red, Navy,Specifications of Alisha Solid Women's Cycling Shorts Shorts Details Number of Contents in Sales Package Pack of 3 Fabric Cotton Lycra Type Cycling Shorts General Details Pattern Solid Ideal For Women's Fabric Care Gentle Machine Wash in Lukewarm Water, Do Not Bleach Additional Details Style Code ALTHT_3P_21 In the Box 3 shorts ["Clothing >> Women's Clothing >> Lingerie, Sleep & Swimwear >> Shorts >> Alisha Shorts >> Alisha Solid Women's Cycling Shorts"] Alisha

Texte nettoyé :
alisha solid women s cycling shorts key features of alisha solid women s cycling shorts cotton lycra navy red navy specifications of alisha solid women s cycling shorts shorts details number of contents in sales package pack of 3 fabric cotton lycra type cycling shorts general details pattern solid ideal for women s fabric care

In [24]:
df_important.to_csv("flipkart_prepared.csv", index=False)

## deuxieme partie 

In [26]:
df_important = df_important.reset_index(drop=True)
df_important["doc_id"] = df.index
df_important.columns

Index(['uniq_id', 'product_name', 'description', 'product_category_tree',
       'brand', 'document_text', 'clean_text', 'tokens', 'final_text',
       'doc_id'],
      dtype='object')

L'ordinateur regarde si ce produit (doc_id) est déjà enregistré pour ce mot.
Si ce n'est pas le cas, il l'initialise à 0.
Ensuite, il ajoute +1 (il compte le nombre de fois où le mot apparaît dans ce produit).

In [28]:
from collections import defaultdict

inverted_index = defaultdict(dict)

for doc_id, tokens in zip(df_important["doc_id"], df_important["tokens"]):
    for token in tokens:
        if doc_id not in inverted_index[token]:
            inverted_index[token][doc_id] = 0
        inverted_index[token][doc_id] += 1

In [34]:
print(inverted_index["phone"])

{327: 1, 491: 2, 607: 16, 991: 4, 1012: 4, 1017: 4, 1024: 4, 1057: 4, 1059: 4, 1081: 6, 1096: 6, 1111: 6, 1114: 6, 1444: 2, 1651: 10, 1658: 10, 1663: 10, 1674: 10, 1691: 10, 1702: 10, 1711: 10, 1756: 10, 1787: 10, 1802: 6, 1853: 10, 1858: 10, 1869: 10, 1876: 10, 1883: 10, 1896: 10, 1921: 4, 1922: 10, 1935: 4, 1936: 4, 1939: 4, 1948: 6, 3124: 1, 4696: 6, 4713: 2, 4716: 2, 4726: 2, 4781: 2, 4826: 2, 4886: 2, 4969: 2, 5005: 2, 5034: 2, 5041: 2, 5053: 6, 5165: 2, 5320: 12, 7649: 1, 7650: 1, 7651: 1, 7652: 1, 7653: 1, 7654: 1, 7655: 1, 7656: 1, 7657: 1, 7659: 1, 7660: 1, 7661: 1, 7663: 1, 7664: 1, 7665: 1, 7667: 1, 7668: 1, 7669: 1, 7693: 9, 7696: 2, 7697: 2, 7699: 2, 7700: 2, 7701: 2, 7702: 2, 7704: 2, 7707: 2, 7708: 2, 7890: 1, 7921: 6, 7993: 1, 9127: 2, 9142: 2, 9152: 2, 9157: 2, 9177: 2, 9186: 2, 9216: 2, 9225: 2, 9229: 2, 9239: 2, 9266: 2, 9272: 2, 9276: 2, 9286: 2, 9287: 2, 9298: 2, 9603: 2, 9644: 2, 9646: 2, 9647: 2, 9648: 2, 9649: 2, 9651: 2, 9652: 2, 9653: 2, 9654: 2, 9655: 2, 9656

In [ ]:
document_lengths = {}

for doc_id, tokens in zip(df_important["doc_id"], df_important["tokens"]):
    document_lengths[doc_id] = len(tokens)
    

In [31]:
print(document_lengths)

{0: 76, 1: 556, 2: 97, 3: 74, 4: 61, 5: 125, 6: 76, 7: 552, 8: 73, 9: 76, 10: 54, 11: 149, 12: 68, 13: 76, 14: 50, 15: 75, 16: 552, 17: 55, 18: 150, 19: 556, 20: 59, 21: 49, 22: 76, 23: 48, 24: 50, 25: 40, 26: 40, 27: 44, 28: 39, 29: 38, 30: 90, 31: 244, 32: 79, 33: 79, 34: 76, 35: 45, 36: 169, 37: 54, 38: 71, 39: 40, 40: 45, 41: 120, 42: 131, 43: 120, 44: 61, 45: 61, 46: 62, 47: 293, 48: 176, 49: 76, 50: 61, 51: 120, 52: 184, 53: 61, 54: 108, 55: 71, 56: 61, 57: 76, 58: 86, 59: 116, 60: 60, 61: 66, 62: 76, 63: 71, 64: 55, 65: 54, 66: 112, 67: 71, 68: 66, 69: 72, 70: 141, 71: 186, 72: 110, 73: 71, 74: 89, 75: 55, 76: 205, 77: 157, 78: 66, 79: 107, 80: 201, 81: 30, 82: 197, 83: 65, 84: 101, 85: 142, 86: 47, 87: 53, 88: 50, 89: 27, 90: 54, 91: 31, 92: 47, 93: 193, 94: 49, 95: 42, 96: 37, 97: 59, 98: 34, 99: 43, 100: 89, 101: 31, 102: 33, 103: 99, 104: 31, 105: 33, 106: 66, 107: 36, 108: 72, 109: 97, 110: 34, 111: 40, 112: 65, 113: 97, 114: 54, 115: 30, 116: 148, 117: 36, 118: 33, 119: 13

In [32]:
tf_index = defaultdict(dict)

for term, postings in inverted_index.items():
    for doc_id, frequency in postings.items():
        tf_index[term][doc_id] = frequency / document_lengths[doc_id]

In [35]:
document_frequency = {}

for term, postings in inverted_index.items():
    document_frequency[term] = len(postings)

In [41]:
print("Le mot shoes apparaît dans 715 produits différents.")
print( document_frequency["shoes"])

Le mot shoes apparaît dans 715 produits différents.
715


IDF(t) = log((N + 1) / (DF(t) + 1)) + 1

N = nombre total de documents
DF(t) = nombre de documents contenant le mot t

In [43]:
import math

N = len(df_important)
idf = {}

for term, df_term in document_frequency.items():
    idf[term] = math.log((N + 1) / (df_term + 1)) + 1

In [54]:
idf["mouse"]

6.613178100138862

TF-IDF(t, d) = TF(t, d) × IDF(t)

In [55]:
tfidf_index = defaultdict(dict)

for term, postings in tf_index.items():
    for doc_id, tf_value in postings.items():
        tfidf_index[term][doc_id] = tf_value * idf[term]

In [56]:
print(tfidf_index["shoes"])

{2: 0.08927747163560015, 17: 0.15745299543005845, 40: 0.48110637492517855, 59: 0.03732721874419489, 89: 0.16036879164172618, 91: 0.13967604433311637, 96: 0.1170258749818002, 98: 0.12735168748019432, 101: 0.13967604433311637, 102: 0.1312108295250487, 104: 0.13967604433311637, 105: 0.1312108295250487, 107: 0.12027659373129464, 109: 0.044638735817800076, 110: 0.12735168748019432, 112: 0.06661472883579396, 113: 0.044638735817800076, 115: 0.1443319124775536, 117: 0.12027659373129464, 118: 0.1312108295250487, 120: 0.1110245480596566, 121: 0.12735168748019432, 122: 0.12735168748019432, 124: 0.12735168748019432, 125: 0.09622127498503572, 127: 0.06098531513136067, 128: 0.13967604433311637, 131: 0.12735168748019432, 133: 0.09622127498503572, 135: 0.11394624669280545, 136: 0.12027659373129464, 139: 0.1443319124775536, 140: 0.12735168748019432, 141: 0.1170258749818002, 142: 0.16036879164172618, 145: 0.1312108295250487, 149: 0.1170258749818002, 151: 0.12735168748019432, 154: 0.12735168748019432, 15

## Préparer la requête utilisateur

In [57]:
def prepare_query(query):
    query = clean_text(query)
    tokens = tokenize(query)
    tokens = remove_stop_words(tokens)
    tokens = remove_short_words(tokens)
    return tokens

In [58]:
def search(query, top_n=10):
    query_tokens = prepare_query(query)

    scores = defaultdict(float)

    for term in query_tokens:
        if term in tfidf_index:
            for doc_id, score in tfidf_index[term].items():
                scores[doc_id] += score

    ranked_results = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    return ranked_results[:top_n]

In [61]:
def show_results(query, top_n=10):
    results = search(query, top_n)

    if len(results) == 0:
        print("Aucun résultat trouvé.")
        return

    print(f"Résultats pour la requête : {query}")
    print("-" * 80)

    for doc_id, score in results:
        product = df_important[df_important["doc_id"] == doc_id].iloc[0]

        print("Score :", round(score, 4))
        print("Nom du produit :", product["product_name"])
        print("Marque :", product["brand"])
        print("Catégorie :", product["product_category_tree"])
        print("Prix :", product.get("discounted_price", "Non disponible"))
        print("-" * 80)

In [62]:
show_results("black shoes", 5)

Résultats pour la requête : black shoes
--------------------------------------------------------------------------------
Score : 1.0392
Nom du produit : Basics Chukka Casual Shoes
Marque : 
Catégorie : ["Footwear >> Men's Footwear >> Casual Shoes >> Basics Casual Shoes"]
Prix : Non disponible
--------------------------------------------------------------------------------
Score : 0.7742
Nom du produit : Lippy Black Lace Up Shoes
Marque : 
Catégorie : ["Footwear >> Men's Footwear >> Formal Shoes >> Lippy Formal Shoes"]
Prix : Non disponible
--------------------------------------------------------------------------------
Score : 0.7217
Nom du produit : Feather Leather Shoes 028 Lace Up Shoes
Marque : 
Catégorie : ["Footwear >> Men's Footwear >> Formal Shoes >> Feather Leather Formal Shoes"]
Prix : Non disponible
--------------------------------------------------------------------------------
Score : 0.7214
Nom du produit : Winkel Black Formal Lace Up Shoes
Marque : 
Catégorie : ["Footwea